# HERMES Phase 2 — Google Colab Free Restore

Runs on **CPU**, mounts your Google Drive, verifies 18 source chunks and the assembled ZIP with SHA-256, and saves a **single verified ZIP** in Drive. No GPU, training, merge, or paid Google Cloud.

**Important:** the Drive folder must be visible under *My Drive* (or a shortcut to it). The ZIP is about 1.66 GB. Colab temporary storage should have at least 4 GB free. This notebook does not verify Android compatibility.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil, json, hashlib, zipfile, os
root = Path('/content/drive/MyDrive')
found = list(root.rglob('HERMES_Drive_Manifest.json'))
print('Manifest candidates:', [str(p) for p in found])
assert len(found) == 1, 'Expected exactly one manifest. Place the folder in My Drive, or edit SOURCE below.'
SOURCE = found[0].parent
OUTPUT = SOURCE / 'HERMES_Phase2_Verified.zip'
manifest = json.loads((SOURCE / 'HERMES_Drive_Manifest.json').read_text())
print('Source:', SOURCE)
print('Expected archive bytes:', manifest['archive_size_bytes'])
print('Free Colab disk GB:', round(shutil.disk_usage('/content').free / 1e9, 2))
assert shutil.disk_usage('/content').free > manifest['archive_size_bytes'] + 1024**3, 'Not enough temporary disk'

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

parts = manifest['parts']
assert len(parts) == 18, 'Unexpected part count'
assert [p['name'] for p in parts] == [f"HERMES_Phase2_Drive.zip.part{i:03d}" for i in range(1, 19)]
assert sum(p['size_bytes'] for p in parts) == manifest['archive_size_bytes']
for part in parts:
    path = SOURCE / part['name']
    assert path.is_file() and not path.is_symlink(), f'Missing or unsafe: {path.name}'
    assert path.stat().st_size == part['size_bytes'], f'Size mismatch: {path.name}'
    assert sha256(path) == part['sha256'], f'SHA-256 mismatch: {path.name}'
    print('PASS', path.name)
print('PASS: all 18 source parts verified')

In [ ]:
temp = Path('/content/HERMES_Phase2_Drive.zip')
with temp.open('wb') as out:
    for part in parts:
        with (SOURCE / part['name']).open('rb') as f:
            shutil.copyfileobj(f, out, 4*1024*1024)
assert temp.stat().st_size == manifest['archive_size_bytes']
assert sha256(temp) == manifest['archive_sha256'], 'Rebuilt ZIP hash mismatch'
with zipfile.ZipFile(temp) as z:
    members = z.infolist()
    expected = {f['name']: f for f in manifest['files']}
    assert len(members) == len(expected)
    assert {m.filename for m in members} == set(expected)
    for m in members:
        assert m.filename == Path(m.filename).name and m.filename not in ('.', '..')
        assert not m.is_dir() and not (m.flag_bits & 1)
        assert m.file_size == expected[m.filename]['size_bytes']
        assert ((m.external_attr >> 16) & 0o170000) != 0o120000
        h = hashlib.sha256()
        with z.open(m) as f:
            for block in iter(lambda: f.read(4*1024*1024), b''):
                h.update(block)
        assert h.hexdigest() == expected[m.filename]['sha256'], f'Inner file mismatch: {m.filename}'
        print('PASS inner:', m.filename)
print('PASS: ZIP and all internal file hashes verified')

In [ ]:
assert not OUTPUT.exists(), f'Refusing to overwrite existing output: {OUTPUT}'
partial = OUTPUT.with_suffix('.zip.partial')
assert not partial.exists(), f'Remove stale partial manually: {partial}'
shutil.copyfile(temp, partial)
assert sha256(partial) == manifest['archive_sha256'], 'Drive copy SHA-256 mismatch'
partial.rename(OUTPUT)
log = SOURCE / 'HERMES_Phase2_Verification.txt'
log.write_text('PASS: 18 parts, ZIP, and all manifest files SHA-256 verified.\n'
               f'archive={OUTPUT.name}\nsha256={manifest["archive_sha256"]}\n'
               f'size_bytes={manifest["archive_size_bytes"]}\n')
print('SUCCESS:', OUTPUT)
print('SHA-256:', manifest['archive_sha256'])
print('Evidence:', log)